# 🃏 GoP3 YOLOv8 Training

Тренировка модели для Governor of Poker 3: карты + dealer button + фолд + банк.

**Порядок действий:**
1. `Runtime → Change runtime type → T4 GPU`
2. Запусти все ячейки по порядку (`Runtime → Run all`)
3. В ячейке **«Загрузка датасета»** загрузи свой zip-архив
4. После тренировки скачай `gop3_model.onnx`

**Ожидаемое время:** ~25–40 минут на T4

## 1. Установка

In [ ]:
!pip install ultralytics onnx onnxruntime -q
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    print('⚠️  GPU не найден — смени Runtime на T4!')

## 2. Загрузка датасета

In [ ]:
import os, zipfile, shutil
from pathlib import Path
from google.colab import files

# Загружаем zip с датасетом
# Ожидаемая структура внутри zip:
#   images/   — все .jpg/.png кадры
#   labels/   — .txt файлы разметки в формате YOLO
#   data.yaml — (опционально, будет создан ниже если нет)

print('Выбери zip-архив с датасетом...')
uploaded = files.upload()
zip_name = list(uploaded.keys())[0]

DATASET_DIR = Path('/content/dataset')
DATASET_DIR.mkdir(exist_ok=True)

with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall(DATASET_DIR)

# Считаем картинки
imgs = list(DATASET_DIR.rglob('*.jpg')) + list(DATASET_DIR.rglob('*.png'))
txts = list(DATASET_DIR.rglob('*.txt'))
print(f'\n✅ Датасет загружен')
print(f'   Картинок: {len(imgs)}')
print(f'   Разметок: {len(txts)}')

## 3. Подготовка структуры датасета

In [ ]:
import random, yaml
from pathlib import Path
import shutil

DATASET_DIR = Path('/content/dataset')
YOLO_DIR    = Path('/content/yolo_data')

for split in ['train', 'val']:
    (YOLO_DIR / 'images' / split).mkdir(parents=True, exist_ok=True)
    (YOLO_DIR / 'labels' / split).mkdir(parents=True, exist_ok=True)

# Собираем все картинки у которых есть разметка
all_imgs = []
for img_path in sorted(DATASET_DIR.rglob('*.jpg')) + sorted(DATASET_DIR.rglob('*.png')):
    lbl_path = img_path.with_suffix('.txt')
    if not lbl_path.exists():
        # Ищем в папке labels/ рядом
        lbl_path2 = img_path.parent.parent / 'labels' / (img_path.stem + '.txt')
        if lbl_path2.exists():
            lbl_path = lbl_path2
        else:
            continue  # нет разметки — пропускаем
    all_imgs.append((img_path, lbl_path))

# 90% train, 10% val
random.seed(42)
random.shuffle(all_imgs)
split_idx = max(1, int(len(all_imgs) * 0.9))
train_set = all_imgs[:split_idx]
val_set   = all_imgs[split_idx:] or all_imgs[:max(1, len(all_imgs)//10)]

for pairs, split in [(train_set, 'train'), (val_set, 'val')]:
    for img_path, lbl_path in pairs:
        shutil.copy(img_path, YOLO_DIR / 'images' / split / img_path.name)
        shutil.copy(lbl_path, YOLO_DIR / 'labels' / split / (img_path.stem + '.txt'))

print(f'Train: {len(train_set)} | Val: {len(val_set)}')

# Классы
CARD_CLASSES = [
    '10C','10D','10H','10S','2C','2D','2H','2S','3C','3D','3H','3S',
    '4C','4D','4H','4S','5C','5D','5H','5S','6C','6D','6H','6S',
    '7C','7D','7H','7S','8C','8D','8H','8S','9C','9D','9H','9S',
    'AC','AD','AH','AS','JC','JD','JH','JS','KC','KD','KH','KS',
    'QC','QD','QH','QS'
]
EXTRA_CLASSES = ['dealer_button','player_folded','player_away','pot_chips','player_bet','stack_label']
ALL_CLASSES   = CARD_CLASSES + EXTRA_CLASSES

# Создаём data.yaml
data_yaml = {
    'path': str(YOLO_DIR),
    'train': 'images/train',
    'val':   'images/val',
    'nc':    len(ALL_CLASSES),
    'names': ALL_CLASSES
}
yaml_path = YOLO_DIR / 'data.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(data_yaml, f, allow_unicode=True, sort_keys=False)

print(f'\n✅ data.yaml записан: {len(ALL_CLASSES)} классов')
print(f'   Карты: 0–51  |  Новые: 52–{len(ALL_CLASSES)-1}')

## 4. Скачиваем базовую модель (playing_cards.onnx → pt для transfer learning)

In [ ]:
# Стартуем с pretrained YOLOv8n на COCO — быстрее сойдётся на новых классах
# Если хочешь transfer learning со старой модели карт — см. комментарий ниже

from ultralytics import YOLO

# Вариант A: стандартный YOLOv8n (рекомендуется если добавляешь новые классы)
base_model = YOLO('yolov8n.pt')
print('✅ Базовая модель загружена: yolov8n.pt')

# Вариант B: загрузить существующую ONNX-модель карт (раскомментировать)
# from google.colab import files
# print('Загрузи playing_cards.onnx...')
# files.upload()
# base_model = YOLO('playing_cards.onnx')  # только инференс, не обучение
# print('Для transfer learning нужен .pt — используй Вариант A')

## 5. Тренировка

In [ ]:
from ultralytics import YOLO
from pathlib import Path

model = YOLO('yolov8n.pt')

results = model.train(
    data    = str(Path('/content/yolo_data/data.yaml')),
    epochs  = 60,          # 60 эпох — хороший баланс скорость/качество
    imgsz   = 640,
    batch   = 16,          # T4 тянет 16 без проблем
    device  = 0,           # GPU
    workers = 2,
    patience= 15,          # early stopping если нет улучшения 15 эпох
    name    = 'gop3',
    project = '/content/runs',

    # Аугментации (помогают при небольшом датасете)
    flipud  = 0.0,         # карты не переворачиваем вертикально
    fliplr  = 0.3,         # горизонтальный флип иногда ок
    mosaic  = 0.8,
    scale   = 0.3,
    hsv_h   = 0.015,
    hsv_s   = 0.5,
    hsv_v   = 0.3,
)

print('\n✅ Тренировка завершена!')
print(f'   mAP50: {results.results_dict.get("metrics/mAP50(B)", "?"):.3f}')

## 6. Экспорт в ONNX

In [ ]:
from ultralytics import YOLO
from pathlib import Path
import glob

# Находим лучший чекпоинт
best_pt = Path('/content/runs/gop3/weights/best.pt')
if not best_pt.exists():
    candidates = glob.glob('/content/runs/**/best.pt', recursive=True)
    best_pt = Path(candidates[0]) if candidates else None

if best_pt is None:
    print('❌ best.pt не найден — тренировка не завершилась?')
else:
    print(f'Лучший чекпоинт: {best_pt}')
    model = YOLO(str(best_pt))
    model.export(
        format    = 'onnx',
        imgsz     = 640,
        opset     = 12,      # совместимо с onnxruntime-web
        simplify  = True,
        dynamic   = False,
    )
    onnx_path = best_pt.with_suffix('.onnx')
    import shutil
    shutil.copy(onnx_path, '/content/gop3_model.onnx')
    print(f'\n✅ ONNX экспортирован → /content/gop3_model.onnx')
    import os
    size_mb = os.path.getsize('/content/gop3_model.onnx') / 1024 / 1024
    print(f'   Размер: {size_mb:.1f} MB')

## 7. Проверка на тестовом изображении (опционально)

In [ ]:
from ultralytics import YOLO
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from google.colab import files

print('Загрузи тестовый скриншот GoP3...')
uploaded = files.upload()
test_img = list(uploaded.keys())[0]

model = YOLO('/content/gop3_model.onnx', task='detect')
res   = model(test_img, conf=0.35)[0]

img = Image.open(test_img)
fig, ax = plt.subplots(1, 1, figsize=(14, 8))
ax.imshow(img)

CARD_CLASSES = [
    '10C','10D','10H','10S','2C','2D','2H','2S','3C','3D','3H','3S',
    '4C','4D','4H','4S','5C','5D','5H','5S','6C','6D','6H','6S',
    '7C','7D','7H','7S','8C','8D','8H','8S','9C','9D','9H','9S',
    'AC','AD','AH','AS','JC','JD','JH','JS','KC','KD','KH','KS',
    'QC','QD','QH','QS'
]
EXTRA_CLASSES = ['dealer_button','player_folded','player_away','pot_chips','player_bet','stack_label']
ALL_CLASSES   = CARD_CLASSES + EXTRA_CLASSES
COLORS = {**{c: '#00ff88' for c in CARD_CLASSES}, **{c: '#ff4444' for c in EXTRA_CLASSES}}

for box in res.boxes:
    x1,y1,x2,y2 = box.xyxy[0].tolist()
    cls  = ALL_CLASSES[int(box.cls)]
    conf = float(box.conf)
    color = COLORS.get(cls, '#ffffff')
    rect = patches.Rectangle((x1,y1), x2-x1, y2-y1,
                               linewidth=2, edgecolor=color, facecolor='none')
    ax.add_patch(rect)
    ax.text(x1, y1-4, f'{cls} {conf:.2f}', color=color, fontsize=8,
            bbox=dict(facecolor='black', alpha=0.5, pad=1))

ax.axis('off')
plt.title(f'Детекций: {len(res.boxes)}')
plt.tight_layout()
plt.savefig('/content/test_result.jpg', dpi=150)
plt.show()
print('Сохранено: /content/test_result.jpg')

## 8. Скачать модель

In [ ]:
from google.colab import files
print('Скачиваю gop3_model.onnx...')
files.download('/content/gop3_model.onnx')
print('\nГотово! Положи файл в:')
print('  artifacts/poker-advisor/public/models/gop3_model.onnx')